In [1]:
import os
import json
from pathlib import Path

import pandas as pd
import serpapi
from dotenv import load_dotenv

print("Libraries imported successfully! ✅")


Libraries imported successfully! ✅


In [2]:
df = pd.read_csv("../data/processed/manali_places_clean.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))
df.head()


Rows: 20
Columns: 10


,place_id,name,address,country,latitude,longitude,rating,reviews,category,data_quality_score
0,ChIJPeVowAaIBDkRnIfuPkskqi0,Hadimba Devi Temple,"Regency Road, Siyal Rd, Siyal, Manali, Himacha...",India,32.248353,77.181573,4.6,49688,Tourist attraction,6
1,ChIJ3VQyL0WHBDkR4Uml16nJto8,Old Manali snow point,"65XJ+J2W, Hadimba Temple Path, Old Manali, Man...",India,32.249112,77.180076,4.6,428,Tourist attraction,6
2,ChIJk9K3np2HBDkRS4FxsgYCMKI,Nehru Kund,"Bashisht, Himachal Pradesh 175103, India",India,32.285982,77.179824,4.4,7767,Tourist attraction,6
3,ChIJJSBNlWOJBDkRkmyIuy0OvGE,Kullu Manali River rafting,"65VQ+7MF, Siyal, Manali, Himachal Pradesh 1751...",India,32.243187,77.189176,4.5,88,Tourist attraction,6
4,ChIJHZ3eboyHBDkRBLrRpkcXmO4,Jogini Falls,"On water fall way V.P.O.-Vashist 5 km.from, Ma...",India,32.275076,77.188146,4.6,10842,Tourist attraction,6


In [3]:
load_dotenv()

api_key = os.getenv("SERPAPI_KEY")

if not api_key:
    raise ValueError("SERPAPI_KEY is not found in .env")

client = serpapi.Client(api_key=api_key)

print("SerpApi client ready! ✅")


SerpApi client ready! ✅


In [4]:
city = "Manali"

category_queries = {
    "nature": f"nature places in {city}",
    "history": f"historical places in {city}",
    "culture": f"cultural places in {city}",
    "adventure": f"adventure activities in {city}",
    "photography": f"photography spots in {city}",
    "shopping": f"shopping places in {city}",
    "religious": f"temples and religious places in {city}",
    "family": f"family friendly places in {city}",
}

category_queries


{'nature': 'nature places in Manali',
 'history': 'historical places in Manali',
 'culture': 'cultural places in Manali',
 'adventure': 'adventure activities in Manali',
 'photography': 'photography spots in Manali',
 'shopping': 'shopping places in Manali',
 'religious': 'temples and religious places in Manali',
 'family': 'family friendly places in Manali'}

In [5]:
cache_dir = Path("../data/raw/enrichment_cache")
cache_dir.mkdir(parents=True, exist_ok=True)

def safe_filename(text):
    return (
        text.lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("&", "and")
    )

def search_with_cache(query, label):
    cache_file = cache_dir / f"{safe_filename(label)}.json"

    if cache_file.exists():
        print(f"📦 Using cache: {cache_file.name}")
        with open(cache_file, "r", encoding="utf-8") as f:
            return json.load(f)

    print(f"🔎 API search: {query}")

    result = client.search({
        "engine": "google_maps",
        "q": query
    })

    result_dict = dict(result)

    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(result_dict, f, indent=4, ensure_ascii=False)

    print(f"💾 Cached: {cache_file.name}")

    return result_dict


In [6]:
enrichment_results = {}

for label, query in category_queries.items():
    enrichment_results[label] = search_with_cache(query, label)

print("\n✅ Enrichment searches completed.")


🔎 API search: nature places in Manali
💾 Cached: nature.json
🔎 API search: historical places in Manali
💾 Cached: history.json
🔎 API search: cultural places in Manali
💾 Cached: culture.json
🔎 API search: adventure activities in Manali
💾 Cached: adventure.json
🔎 API search: photography spots in Manali
💾 Cached: photography.json
🔎 API search: shopping places in Manali
💾 Cached: shopping.json
🔎 API search: temples and religious places in Manali
💾 Cached: religious.json
🔎 API search: family friendly places in Manali
💾 Cached: family.json

✅ Enrichment searches completed.


In [7]:
for label, result in enrichment_results.items():
    places = result.get("local_results", [])
    print(f"{label:12} → {len(places)} results")


nature       → 20 results
history      → 20 results
culture      → 4 results
adventure    → 20 results
photography  → 20 results
shopping     → 14 results
religious    → 20 results
family       → 20 results


In [8]:
place_signals = {}

for label, result in enrichment_results.items():
    for place in result.get("local_results", []):
        place_id = place.get("place_id")

        if not place_id:
            continue

        if place_id not in place_signals:
            place_signals[place_id] = set()

        place_signals[place_id].add(label)

print("Unique places found during enrichment:", len(place_signals))


Unique places found during enrichment: 99


In [9]:
feature_names = list(category_queries.keys())

for feature in feature_names:
    df[feature] = df["place_id"].apply(
        lambda pid: int(feature in place_signals.get(pid, set()))
    )

df[[
    "name",
    *feature_names
]].head(20)


,name,nature,history,culture,adventure,photography,shopping,religious,family
0,Hadimba Devi Temple,1,1,1,0,1,1,1,0
1,Old Manali snow point,1,1,1,0,1,0,0,1
2,Nehru Kund,0,1,0,0,1,0,0,0
3,Kullu Manali River rafting,0,0,0,0,0,0,0,0
4,Jogini Falls,1,1,0,0,1,0,0,1
5,Van Vihar National Park,1,1,0,0,1,0,0,1
6,Manali View Point,0,0,0,0,1,0,0,0
7,Rahala Waterfalls,1,1,0,0,0,0,0,1
8,Lama Dugh Trek Start Point,1,0,0,0,0,0,0,0
9,Atal Bihari statue,0,1,0,0,0,0,0,0


In [10]:
def get_tags(place_id):
    signals = place_signals.get(place_id, set())
    return ", ".join(sorted(signals))

df["travel_tags"] = df["place_id"].apply(get_tags)

df[["name", "travel_tags"]].head(20)


,name,travel_tags
0,Hadimba Devi Temple,"culture, history, nature, photography, religio..."
1,Old Manali snow point,"culture, family, history, nature, photography"
2,Nehru Kund,"history, photography"
3,Kullu Manali River rafting,
4,Jogini Falls,"family, history, nature, photography"
5,Van Vihar National Park,"family, history, nature, photography"
6,Manali View Point,photography
7,Rahala Waterfalls,"family, history, nature"
8,Lama Dugh Trek Start Point,nature
9,Atal Bihari statue,history


In [11]:
signal_counts = df[feature_names].sum().sort_values(ascending=False)
signal_counts


history        13
nature         10
family          8
photography     6
shopping        2
culture         2
religious       2
adventure       0
dtype: int64

In [12]:
df["interest_count"] = df[feature_names].sum(axis=1)

df[[
    "name",
    "travel_tags",
    "interest_count"
]].sort_values(
    "interest_count",
    ascending=False
).head(15)


,name,travel_tags,interest_count
0,Hadimba Devi Temple,"culture, history, nature, photography, religio...",6
1,Old Manali snow point,"culture, family, history, nature, photography",5
5,Van Vihar National Park,"family, history, nature, photography",4
4,Jogini Falls,"family, history, nature, photography",4
19,Manali Bazaar,"family, history, nature, shopping",4
7,Rahala Waterfalls,"family, history, nature",3
18,Gulaba Viewpoint,"family, history, nature",3
2,Nehru Kund,"history, photography",2
15,Old Manali View point,"family, history",2
12,Shiv Mahadev Temple,"history, religious",2


In [13]:
df[[
    "name",
    "category",
    "travel_tags",
    "nature",
    "history",
    "culture",
    "adventure",
    "photography",
    "shopping",
    "religious",
    "family"
]].head(20)


,name,category,travel_tags,nature,history,culture,adventure,photography,shopping,religious,family
0,Hadimba Devi Temple,Tourist attraction,"culture, history, nature, photography, religio...",1,1,1,0,1,1,1,0
1,Old Manali snow point,Tourist attraction,"culture, family, history, nature, photography",1,1,1,0,1,0,0,1
2,Nehru Kund,Tourist attraction,"history, photography",0,1,0,0,1,0,0,0
3,Kullu Manali River rafting,Tourist attraction,,0,0,0,0,0,0,0,0
4,Jogini Falls,Tourist attraction,"family, history, nature, photography",1,1,0,0,1,0,0,1
5,Van Vihar National Park,Tourist attraction,"family, history, nature, photography",1,1,0,0,1,0,0,1
6,Manali View Point,Tourist attraction,photography,0,0,0,0,1,0,0,0
7,Rahala Waterfalls,Tourist attraction,"family, history, nature",1,1,0,0,0,0,0,1
8,Lama Dugh Trek Start Point,Tourist attraction,nature,1,0,0,0,0,0,0,0
9,Atal Bihari statue,Tourist attraction,history,0,1,0,0,0,0,0,0


In [14]:
output_path = "../data/processed/manali_places_enriched.csv"

df.to_csv(output_path, index=False)

print(f"✅ Enriched dataset saved to: {output_path}")
print("Final shape:", df.shape)


✅ Enriched dataset saved to: ../data/processed/manali_places_enriched.csv
Final shape: (20, 20)
